In [ ]:
import copy
import torch
import numpy as np
from detectron2.data import detection_utils as utils
from detectron2.data import transforms as T
from detectron2.structures import Instances, BoxMode

class CustomDrivingMapper:
    """
    사용자 정의 Mapper:
    1. 이미지 및 2D Annotation(Polygon) Augmentation 적용
    2. 3D 정보(Distance 등)의 None 값을 처리하여 Tensor 및 Mask 생성
    """
    def __init__(self, cfg, is_train=True):
        self.is_train = is_train
        
        # 기본적인 Augmentation 설정 (Resize, Flip 등)
        # 실제 학습 시에는 cfg.INPUT.MIN_SIZE_TRAIN 등을 참조하도록 수정 가능
        if is_train:
            self.tfm_gens = [
                T.ResizeShortestEdge([800, 800], 1333),
                T.RandomFlip(prob=0.5, horizontal=True, vertical=False),
            ]
        else:
            self.tfm_gens = [T.ResizeShortestEdge([800, 800], 1333)]

        # 이미지 포맷 설정
        self.img_format = cfg.INPUT.FORMAT

    def __call__(self, dataset_dict):
        """
        dataset_dict: 앞서 작성한 get_filtered_fusion_dataset에서 리턴된 dict 1개
        """
        dataset_dict = copy.deepcopy(dataset_dict)
        
        # 1. 이미지 읽기
        image = utils.read_image(dataset_dict["file_name"], format=self.img_format)
        utils.check_image_size(dataset_dict, image)

        # 2. Transform 적용 (이미지 & 좌표 변환)
        image, transforms = T.apply_transform_gens(self.tfm_gens, image)
        image_shape = image.shape[:2]  # h, w

        # 이미지를 텐서로 변환 (C, H, W)
        dataset_dict["image"] = torch.as_tensor(np.ascontiguousarray(image.transpose(2, 0, 1)))

        # 3. Annotations 처리
        if "annotations" not in dataset_dict:
            return dataset_dict

        # 좌표 변환을 위해 deepcopy
        annos = [
            utils.transform_instance_annotations(
                obj, transforms, image_shape
            )
            for obj in dataset_dict.pop("annotations")
            if obj.get("iscrowd", 0) == 0
        ]

        # Instances 객체 생성 (이미지 크기 정보 포함)
        instances = utils.annotations_to_instances(
            annos, image_shape, mask_format="polygon" # bitmask가 아니라면 polygon
        )

        # ============================================================
        # [핵심] 3D 정보(Distance) Tensor 변환 및 Masking 처리
        # ============================================================
        # annos 리스트 순서와 instances 내부 순서는 동일함이 보장됨
        
        distances = []
        distance_masks = []
        
        # 추가 정보들도 필요하면 같은 방식으로 처리
        # locations = [] 
        # dimensions = []

        for anno in annos:
            dist = anno.get("distance")
            
            if dist is not None:
                # 값이 있으면: 값 그대로 넣고, 마스크는 True(1)
                distances.append(float(dist))
                distance_masks.append(True)
            else:
                # 값이 없으면(None): 0.0(더미) 넣고, 마스크는 False(0)
                distances.append(0.0)
                distance_masks.append(False)

        # 리스트를 텐서로 변환하여 Instances 객체에 새로운 필드로 등록
        # (N, ) 형태의 1차원 텐서
        instances.gt_distances = torch.tensor(distances, dtype=torch.float32)
        instances.gt_distance_masks = torch.tensor(distance_masks, dtype=torch.bool)

        # (참고) Location이나 Dimension은 (N, 3) 형태일 것이므로
        # 없으면 [0.0, 0.0, 0.0] 등으로 채워야 함.
        
        dataset_dict["instances"] = utils.filter_empty_instances(instances)
        
        return dataset_dict

In [ ]:
from detectron2.engine import DefaultTrainer
from detectron2.data import build_detection_train_loader 

class MyCustomTrainer(DefaultTrainer):
    """
    기본 Trainer를 상속받아 데이터 로더 부분만 커스터마이징합니다.
    """
    @classmethod
    def build_train_loader(cls, cfg):
        # 여기서 우리가 만든 Mapper를 주입합니다.
        mapper = CustomDrivingMapper(cfg, is_train=True)
        
        # 데이터셋 이름은 cfg.DATASETS.TRAIN[0] 등을 참조하여 로더가 알아서 가져옵니다.
        return build_detection_train_loader(cfg, mapper=mapper)

In [ ]:
import os
import json
import copy
import numpy as np
import torch

from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer, default_setup
from detectron2.data import DatasetCatalog, MetadataCatalog, detection_utils as utils
from detectron2.data import transforms as T
from detectron2.structures import BoxMode, Instances
from detectron2.data import build_detection_train_loader

# ==============================================================================
# [1] 사용자 설정: 클래스 및 경로
# ==============================================================================

JSON_2D_DIR = '/content/drive/MyDrive/AD/Sesac/projects/first_project/Sample/labels/2D/08_174514_221206/sensor_raw_data/camera'
JSON_3D_DIR = '/content/drive/MyDrive/AD/Sesac/projects/first_project/Sample/labels/3D/08_174514_221206/sensor_raw_data/camera'
IMG_DIR = '/content/drive/MyDrive/AD/Sesac/projects/first_project/Sample/images/08_174514_221206/sensor_raw_data/camera'


THING_CLASSES = [
    "vehicle", "bus", "truck", "otherCar",
    "motorcycle", "bicycle", "pedestrian", "rider",
    "TrafficSign", "TrafficLight", "constructionGuide", "trafficDrum", 
]
STUFF_CLASSES = [
    "Freespace", "curb", "sidewalk", "crosswalk", 
    "roadMark", "whiteLane", "yellowLane", 
]

ALL_CLASSES = THING_CLASSES + STUFF_CLASSES
TARGET_SET = set(ALL_CLASSES)
CLASS_TO_ID = {name: i for i, name in enumerate(ALL_CLASSES)}

# ==============================================================================
# [2] 데이터셋 로더 (Filter & Match Logic)
# ==============================================================================
def get_bbox_from_poly(poly):
    xs = poly[0::2]; ys = poly[1::2]
    return [min(xs), min(ys), max(xs), max(ys)]

def compute_iou(box_a, box_b):
    xA = max(box_a[0], box_b[0]); yA = max(box_a[1], box_b[1])
    xB = min(box_a[2], box_b[2]); yB = min(box_a[3], box_b[3])
    inter_area = max(0, xB - xA) * max(0, yB - yA)
    box_a_area = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    box_b_area = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union = float(box_a_area + box_b_area - inter_area)
    return inter_area / union if union > 0 else 0.0

def get_filtered_fusion_dataset(json_dir_2d, json_dir_3d, image_dir):
    dataset_dicts = []
    files = [f for f in os.listdir(json_dir_2d) if f.endswith(".json")]
    
    for idx, filename in enumerate(files):
        path_2d = os.path.join(json_dir_2d, filename)
        path_3d = os.path.join(json_dir_3d, filename)
        
        with open(path_2d) as f: data_2d = json.load(f)
        candidates_3d = []
        if os.path.exists(path_3d):
            with open(path_3d) as f: candidates_3d = json.load(f).get("annotations", [])

        record = {}
        # [중요] 1920x1080 해상도 정보 설정
        info = data_2d["information"]
        record["file_name"] = os.path.join(image_dir, info["filename"])
        record["image_id"] = idx
        record["height"] = 1080 # JSON에 있지만 명시적으로 고정 (안전장치)
        record["width"] = 1920
        
        objs = []
        for item_2d in data_2d["annotations"]:
            cls_name = item_2d["class"]
            if cls_name not in TARGET_SET: continue
            
            poly = item_2d["polygon"]
            poly_bbox = get_bbox_from_poly(poly)
            
            obj = {
                "bbox": poly_bbox, "bbox_mode": BoxMode.XYXY_ABS,
                "category_id": CLASS_TO_ID[cls_name],
                "segmentation": [poly],
                "distance": None # Default
            }
            
            # 매칭 로직
            best_iou = 0.0
            best_match = None
            for item_3d in candidates_3d:
                if item_3d["class"] != cls_name: continue
                iou = compute_iou(poly_bbox, item_3d["bbox"])
                if iou > 0.5 and iou > best_iou:
                    best_iou = iou
                    best_match = item_3d
            
            if best_match:
                obj["distance"] = best_match.get("distance")
                # obj["dimension"] = ... 필요시 추가
            
            objs.append(obj)
        record["annotations"] = objs
        dataset_dicts.append(record)
    return dataset_dicts

# ==============================================================================
# [3] Custom Mapper (1920x1080 최적화)
# ==============================================================================
class CustomDrivingMapper:
    def __init__(self, cfg, is_train=True):
        self.is_train = is_train
        self.img_format = cfg.INPUT.FORMAT
        
        # [해상도 전략] 1920x1080 입력 -> Resize
        # Short Edge를 800으로 줄이면 -> 800x1422가 됨.
        # 하지만 Max Size가 1333이면 -> 750x1333으로 조정됨.
        # VRAM 여유가 많다면 max_size를 1920이나 1440으로 늘려도 됨.
        if is_train:
            self.tfm_gens = [
                # 여기서 (800,)은 줄였을 때의 짧은 변 길이 목표값
                T.ResizeShortestEdge(short_edge_length=(800,), max_size=1333, sample_style="choice"),
                T.RandomFlip(prob=0.5, horizontal=True, vertical=False),
            ]
        else:
            self.tfm_gens = [
                T.ResizeShortestEdge(short_edge_length=(800,), max_size=1333, sample_style="choice")
            ]

    def __call__(self, dataset_dict):
        dataset_dict = copy.deepcopy(dataset_dict)
        image = utils.read_image(dataset_dict["file_name"], format=self.img_format)
        utils.check_image_size(dataset_dict, image)

        image, transforms = T.apply_transform_gens(self.tfm_gens, image)
        image_shape = image.shape[:2] 
        dataset_dict["image"] = torch.as_tensor(np.ascontiguousarray(image.transpose(2, 0, 1)))

        if "annotations" not in dataset_dict:
            return dataset_dict

        annos = [
            utils.transform_instance_annotations(obj, transforms, image_shape)
            for obj in dataset_dict.pop("annotations")
            if obj.get("iscrowd", 0) == 0
        ]
        
        instances = utils.annotations_to_instances(annos, image_shape, mask_format="polygon")
        
        # Distance & Mask Tensor 생성
        distances = []
        masks = []
        for anno in annos:
            d = anno.get("distance")
            if d is not None:
                distances.append(float(d))
                masks.append(True)
            else:
                distances.append(0.0)
                masks.append(False)
        
        instances.gt_distances = torch.tensor(distances, dtype=torch.float32)
        instances.gt_distance_masks = torch.tensor(masks, dtype=torch.bool)
        
        dataset_dict["instances"] = utils.filter_empty_instances(instances)
        return dataset_dict

# ==============================================================================
# [4] Custom Trainer & Main
# ==============================================================================
class MyCustomTrainer(DefaultTrainer):
    @classmethod
    def build_train_loader(cls, cfg):
        mapper = CustomDrivingMapper(cfg, is_train=True)
        return build_detection_train_loader(cfg, mapper=mapper)

def main():
    # 1. 데이터 등록
    DatasetCatalog.register("driving_train", lambda: get_filtered_fusion_dataset(JSON_2D_DIR, JSON_3D_DIR, IMG_DIR))
    MetadataCatalog.get("driving_train").set(thing_classes=ALL_CLASSES)
    
    # 2. Config 설정
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
    
    cfg.DATASETS.TRAIN = ("driving_train",)
    cfg.DATASETS.TEST = ()
    cfg.DATALOADER.NUM_WORKERS = 2
    
    # [해상도 설정] Mapper와 동일하게 맞추는 것이 좋습니다.
    # 1920x1080 이미지를 처리하기 위해 메모리를 고려한 사이즈
    cfg.INPUT.MIN_SIZE_TRAIN = (800,)
    cfg.INPUT.MAX_SIZE_TRAIN = 1333
    
    # 학습 파라미터
    cfg.SOLVER.IMS_PER_BATCH = 2 # VRAM 부족 시 1로 줄이세요
    cfg.SOLVER.BASE_LR = 0.001
    cfg.SOLVER.MAX_ITER = 1000
    cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128
    cfg.MODEL.ROI_HEADS.NUM_CLASSES = len(ALL_CLASSES)
    
    cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")
    cfg.OUTPUT_DIR = "./output_multitask"
    os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
    
    trainer = MyCustomTrainer(cfg)
    trainer.resume_or_load(resume=False)
    trainer.train()

if __name__ == "__main__":
    main()